In [1]:
function range(n: number): number[] {
    return Array.from({ length: n + 1 }, (_, i) => i);
}

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes, 
- transform the formula into *prenex normal form*,
  i.e. we move the quantifieres outside,
  
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are 
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall X: q(X)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $X$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $X$ that are bound by the second quantifier as follows:
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall Y: q(Y)\bigr)$$

## Auxilliary Functions

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the parser that is found in the module `FOL-Parser`.  Our parser distinguishes variables and function symbol as follows:
- A word starting with an *upper* case letter is interpreted as a *variable*.
- A word starting with a *lower* case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
import { parseFormula as parse, Formula, Term, Variable, PredicateSymbol } from "./FOL-Parser";
import { Tuple, RecursiveSet as Set, Value } from "recursive-set";

In [3]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [4]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [5]:
const s = '∀G:∀C:(grandparent(G, C) ↔ ∃P: (parent(G, P) ∧ parent(P, C)))';
const f1 = parse(s);
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ '⚛️', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


A `Substitution` maps variables to terms. 

In [6]:
type Substitution = Map<Variable, Term>;

The function $\texttt{apply}(t, σ)$ takes an object $t$ and a *variable substitution* $\sigma$ which is represented as a dictionary of the form $\{X_1: s_1, \cdots, X_n:s_n\}$ and replaces every occurrence of the variable $X_i$ in the object $t$ with the corresponding term $s_i$.  The object $t$ is either 
 - a term, or
 - a formula from first order logic (henceforth abbreviated as *FOL*).

In [7]:
function applyTerm(t: Term, sigma: Substitution): Term {
    if (typeof t === 'string') {
        const mapped = sigma.get(t);
        return mapped !== undefined ? mapped : t;
    }
    const [f, ...args] = t;
    return [f, ...args.map(arg => applyTerm(arg, sigma))];
}

function applyFormula(f: Formula, sigma: Substitution): Formula {
    switch (f[0]) {
        case '⚛️': {
            const [tag, pred, ...args] = f;
            return [tag, pred, ...args.map(arg => applyTerm(arg, sigma))];
        }
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, applyFormula(g, sigma)];
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            return [op, applyFormula(g, sigma), applyFormula(h, sigma)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            const mapped = sigma.get(x);
            const newX = typeof mapped === 'string' ? mapped : x;
            return [op, newX, applyFormula(g, sigma)];
        }
    }
}

In [8]:
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ 'Atom', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


In [9]:
const sigma1: Substitution = new Map([
    ['G', 'X'],
    ['P', 'Y'],
    ['C', 'Z']
]);
console.dir(applyFormula(f1, sigma1), { depth: null });

[
  '∀',
  'X',
  [
    '∀',
    'Z',
    [
      '↔',
      [ '⚛️', 'grandparent', 'X', 'Z' ],
      [
        '∃',
        'Y',
        [
          '∧',
          [ '⚛️', 'parent', 'X', 'Y' ],
          [ '⚛️', 'parent', 'Y', 'Z' ]
        ]
      ]
    ]
  ]
]


The function $\texttt{boundVariables}(f)$ computes the set of variables that are *bound* in the formula $f$. 

In [10]:
function boundVariables(f: Formula): Set<string> {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return set<string>();
        case '¬': {
            const [_, g] = f;
            return boundVariables(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [_, g, h] = f;
            return boundVariables(g).union(boundVariables(h));
        }
        case '∀':
        case '∃': {
            const [_, x, g] = f;
            return boundVariables(g).union(set(x));
        }
    }
}

In [11]:
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ '⚛️', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


In [12]:
console.log([...boundVariables(f1)]);

[ 'P', 'C', 'G' ]


The function `allVariables` computes the set of all variables that occur in terms inside `f`. The object 
`f` is either a formula or a term.

In [13]:
function allVariablesTerm(t: Term): Set<string> {
    if (typeof t === 'string') return set(t);
    const [_, ...args] = t;
    return args.reduce((acc, arg) => acc.union(allVariablesTerm(arg)), set<string>());
}

function allVariables(f: Formula): Set<string> {
    switch(f[0]) {
        case '⚛️': {
            const [_, pred, ...args] = f;
            return args.reduce((acc, arg) => acc.union(allVariablesTerm(arg)), set<string>());
        }
        case '⊤':
        case '⊥':
            return set<string>();
        case '¬': {
            const [_, g] = f;
            return allVariables(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [_, g, h] = f;
            return allVariables(g).union(allVariables(h));
        }
        case '∀':
        case '∃': {
            const [_, x, g] = f;
            return allVariables(g).union(set(x));
        }
    }
}

In [14]:
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ '⚛️', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


In [15]:
console.log([...allVariables(f1)]);

[ 'G', 'C', 'P' ]


In [16]:
const g1: Formula = ['↔', 
    ['⚛️', 'grandparent', 'G', 'C'],
    ['∃', 'P', ['∧', ['⚛️', 'parent', 'G', 'P'], ['⚛️', 'parent', 'P', 'C']]]
];

In [17]:
console.log([...allVariables(g1)]);

[ 'G', 'C', 'P' ]


Below we construct a list of all upper case characters to generate new variables.

In [18]:
const ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".split("");
ascii_uppercase

[
  'A', 'B', 'C', 'D', 'E', 'F',
  'G', 'H', 'I', 'J', 'K', 'L',
  'M', 'N', 'O', 'P', 'Q', 'R',
  'S', 'T', 'U', 'V', 'W', 'X',
  'Y', 'Z'
]


In [19]:
const ascii_set = set(...ascii_uppercase);
console.log(ascii_set.size);

26


The function $\texttt{renameBoundVariables}(f)$ takes a first order formula $f$ and replaces all bound variables by **new** variables.  This only works if the set of characters `set(string.ascii_uppercase)` has enough characters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `X1`, `X2`, `X3`, $\cdots$, but that becomes unreadable very fast.

In [20]:
function renameBoundVariables(f: Formula): Formula {
    const boundVs = [...boundVariables(f)];
    const allVs   = allVariables(f);
    const newVars = ascii_uppercase.filter(x => !allVs.has(x)).sort();
    const mappingPairs = boundVs.map((bv, i) => [bv, newVars[i]] as const);
    const sigma: Substitution = new Map(mappingPairs);    
    return applyFormula(f, sigma);
}

In [21]:
console.log(['A', 'B', 'C'].map((x, i) => [i, x]));

[ [ 0, 'A' ], [ 1, 'B' ], [ 2, 'C' ] ]


In [22]:
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ '⚛️', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


In [23]:
console.dir(renameBoundVariables(f1), { depth: null });

[
  '∀',
  'D',
  [
    '∀',
    'B',
    [
      '↔',
      [ '⚛️', 'grandparent', 'D', 'B' ],
      [
        '∃',
        'A',
        [
          '∧',
          [ '⚛️', 'parent', 'D', 'A' ],
          [ '⚛️', 'parent', 'A', 'B' ]
        ]
      ]
    ]
  ]
]


## Elimination Biconditionals

The function $\texttt{eliminateBiconditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$(f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f)$$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the formula $g \rightarrow f$.

In [24]:
function eliminateBiconditional(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, eliminateBiconditional(g)];
        }
        case '∧':
        case '∨':
        case '→': {
            const [op, g, h] = f;
            return [op, eliminateBiconditional(g), eliminateBiconditional(h)];
        }
        case '↔': {
            const [_, g, h] = f;
            const ge = eliminateBiconditional(g);
            const he = eliminateBiconditional(h);
            const left: Formula = ['→', ge, he];
            const right = renameBoundVariables(['→', he, ge]);
            return ['∧', left, right];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, eliminateBiconditional(g)];
        }
    }
}

In [25]:
console.dir(f1, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '↔',
      [ '⚛️', 'grandparent', 'G', 'C' ],
      [
        '∃',
        'P',
        [
          '∧',
          [ '⚛️', 'parent', 'G', 'P' ],
          [ '⚛️', 'parent', 'P', 'C' ]
        ]
      ]
    ]
  ]
]


In [26]:
const f2 = eliminateBiconditional(f1);
console.dir(f2, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '→',
        [ '⚛️', 'grandparent', 'G', 'C' ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '→',
        [
          '∃',
          'A',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'A' ],
            [ '⚛️', 'parent', 'A', 'C' ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


## Eliminating Conditionals

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$(f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g)$$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.

In [27]:
function eliminateConditional(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [op, g] = f;
            return [op, eliminateConditional(g)];
        }
        case '∧':
        case '∨':
        case '↔': {
            const [op, g, h] = f;
            return [op, eliminateConditional(g), eliminateConditional(h)];
        }
        case '→': {
            const [_, g, h] = f;
            return ['∨', ['¬', eliminateConditional(g)], eliminateConditional(h)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, eliminateConditional(g)];
        }
    }
}

In [28]:
console.dir(f2, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '→',
        [ '⚛️', 'grandparent', 'G', 'C' ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '→',
        [
          '∃',
          'A',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'A' ],
            [ '⚛️', 'parent', 'A', 'C' ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


In [29]:
const f3 = eliminateConditional(f2);
console.dir(f3, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '∨',
        [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '∨',
        [
          '¬',
          [
            '∃',
            'A',
            [
              '∧',
              [ '⚛️', 'parent', 'G', 'A' ],
              [ '⚛️', 'parent', 'A', 'C' ]
            ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


## Negation Normal Form

The function $\texttt{nnf}(f)$ computes the <em style="color:blue;">negation normal form</em> of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$. 

In [30]:
function nnf(f: Formula): Formula {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
            return f;
        case '¬': {
            const [_, g] = f;
            return neg(g);
        }
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            return [op, nnf(g), nnf(h)];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            return [op, x, nnf(g)];
        }
    }
}

function neg(f: Formula): Formula {
    switch (f[0]) {
        case '⊤': return ['⊥'];
        case '⊥': return ['⊤'];
        case '¬': {
            const [_, g] = f;
            return nnf(g);
        }
        case '∧': {
            const [_, g, h] = f;
            return ['∨', neg(g), neg(h)];
        }
        case '∨': {
            const [_, g, h] = f;
            return ['∧', neg(g), neg(h)];
        }
        case '→':
        case '↔':
        case '⚛️':
            return ['¬', f];
        case '∀': {
            const [_, x, g] = f;
            return ['∃', x, neg(g)];
        }
        case '∃': {
            const [_, x, g] = f;
            return ['∀', x, neg(g)];
        }
    }
}

In [31]:
console.dir(f3, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '∨',
        [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '∨',
        [
          '¬',
          [
            '∃',
            'A',
            [
              '∧',
              [ '⚛️', 'parent', 'G', 'A' ],
              [ '⚛️', 'parent', 'A', 'C' ]
            ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


In [32]:
const f4 = nnf(f3);
console.dir(f4, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '∨',
        [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '∨',
        [
          '∀',
          'A',
          [
            '∨',
            [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
            [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


## Prenex Normal Form

In [33]:
type QuantifierList = string[];

function mergeQuantifiers(Q1: QuantifierList, Q2: QuantifierList): QuantifierList {
    if (Q1.length === 0) return Q2;
    if (Q2.length === 0) return Q1;
    if (Q1[0] === '∃') return [Q1[0], Q1[1], ...mergeQuantifiers(Q1.slice(2), Q2)];
    if (Q2[0] === '∃') return [Q2[0], Q2[1], ...mergeQuantifiers(Q1, Q2.slice(2))];
    return [Q1[0], Q1[1], ...mergeQuantifiers(Q1.slice(2), Q2)];
}

In [34]:
console.log(mergeQuantifiers(['∀', 'X', '∃', 'Y'], ['∃', 'U', '∀', 'V']));

[
  '∃', 'U', '∀',
  'X', '∃', 'Y',
  '∀', 'V'
]


In [35]:
function extractQuantifiers(f: Formula): [QuantifierList, Formula] {
    switch (f[0]) {
        case '⚛️':
        case '⊤':
        case '⊥':
        case '¬':
            return [[], f];
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [op, g, h] = f;
            const [qg, gm] = extractQuantifiers(g);
            const [qh, hm] = extractQuantifiers(h);
            return [mergeQuantifiers(qg, qh), [op, gm, hm]];
        }
        case '∀':
        case '∃': {
            const [op, x, g] = f;
            const [qg, gm] = extractQuantifiers(g);
            return [[op, x, ...qg], gm];
        }
    }
}

In [36]:
console.dir(f4, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∧',
      [
        '∨',
        [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
        [
          '∃',
          'P',
          [
            '∧',
            [ '⚛️', 'parent', 'G', 'P' ],
            [ '⚛️', 'parent', 'P', 'C' ]
          ]
        ]
      ],
      [
        '∨',
        [
          '∀',
          'A',
          [
            '∨',
            [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
            [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
          ]
        ],
        [ '⚛️', 'grandparent', 'G', 'C' ]
      ]
    ]
  ]
]


In [37]:
const [Qs, f5] = extractQuantifiers(f4);
console.log(Qs);
console.dir(f5, { depth: null });

[
  '∀', 'G', '∀',
  'C', '∃', 'P',
  '∀', 'A'
]
[
  '∧',
  [
    '∨',
    [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
    [
      '∧',
      [ '⚛️', 'parent', 'G', 'P' ],
      [ '⚛️', 'parent', 'P', 'C' ]
    ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
      [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
    ],
    [ '⚛️', 'grandparent', 'G', 'C' ]
  ]
]


In [38]:
function attachQuantifiers(Qs: QuantifierList, m: Formula): Formula {
    if (Qs.length === 0) return m;
    const Q = Qs[0] as '∀' | '∃';
    const x = Qs[1];
    return [Q, x, attachQuantifiers(Qs.slice(2), m)];
}

In [39]:
console.log(Qs);

[
  '∀', 'G', '∀',
  'C', '∃', 'P',
  '∀', 'A'
]


In [40]:
console.dir(f5, { depth: null });

[
  '∧',
  [
    '∨',
    [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
    [
      '∧',
      [ '⚛️', 'parent', 'G', 'P' ],
      [ '⚛️', 'parent', 'P', 'C' ]
    ]
  ],
  [
    '∨',
    [
      '∨',
      [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
      [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
    ],
    [ '⚛️', 'grandparent', 'G', 'C' ]
  ]
]


In [41]:
const f6 = attachQuantifiers(Qs, f5);
console.dir(f6, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∃',
      'P',
      [
        '∀',
        'A',
        [
          '∧',
          [
            '∨',
            [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
            [
              '∧',
              [ '⚛️', 'parent', 'G', 'P' ],
              [ '⚛️', 'parent', 'P', 'C' ]
            ]
          ],
          [
            '∨',
            [
              '∨',
              [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
              [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
            ],
            [ '⚛️', 'grandparent', 'G', 'C' ]
          ]
        ]
      ]
    ]
  ]
]


## Skolemization (Eliminating Existential Quantifiers)

In [42]:
let skolemCounter = 0;

In [43]:
function skolemConstant(): string {
    skolemCounter += 1;
    return 'sk' + skolemCounter.toString();
}

In [44]:
function skolemize(f: Formula, Vs: string[]): Formula {
    switch (f[0]) {
        case '∃': {
            const [_, x, g] = f;
            const t: Term = [skolemConstant(), ...Vs];
            const sigma: Substitution = new Map();
            sigma.set(x, t);
            return skolemize(applyFormula(g, sigma), Vs);
        }
        case '∀': {
            const [op, x, g] = f;
            return [op, x, skolemize(g, [...Vs, x])];
        }
        default:
            return f;
    }
}

In [45]:
console.dir(f6, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∃',
      'P',
      [
        '∀',
        'A',
        [
          '∧',
          [
            '∨',
            [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
            [
              '∧',
              [ '⚛️', 'parent', 'G', 'P' ],
              [ '⚛️', 'parent', 'P', 'C' ]
            ]
          ],
          [
            '∨',
            [
              '∨',
              [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
              [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
            ],
            [ '⚛️', 'grandparent', 'G', 'C' ]
          ]
        ]
      ]
    ]
  ]
]


In [46]:
const f7 = skolemize(f6, []);
console.dir(f7, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∀',
      'A',
      [
        '∧',
        [
          '∨',
          [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
          [
            '∧',
            [ '⚛️', 'parent', 'G', [ 'sk1', 'G', 'C' ] ],
            [ '⚛️', 'parent', [ 'sk1', 'G', 'C' ], 'C' ]
          ]
        ],
        [
          '∨',
          [
            '∨',
            [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
            [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
          ],
          [ '⚛️', 'grandparent', 'G', 'C' ]
        ]
      ]
    ]
  ]
]


## Conversion to Clauses

In [47]:
// Helper to convert array structure to recursive-set Tuple for structural equality
function toTuple(f: any): any {
    if (typeof f === 'string') return f;
    if (Array.isArray(f)) return tpl(...f.map(toTuple));
    return f;
}

type Literal = Tuple<any>;
type Clause = Set<Literal>;
type CNFSet = Set<Clause>;

function cnf(f: Formula): CNFSet {
    switch (f[0]) {
        case '⊤': return set<Clause>();
        case '⊥': return set<Clause>(set<Literal>());
        case '¬': return set<Clause>(set<Literal>(toTuple(f)));
        case '∧': {
            const [_, g, h] = f;
            return cnf(g).union(cnf(h));
        }
        case '∨': {
            const [_, g, h] = f;
            const cnfg = cnf(g);
            const cnfh = cnf(h);
            const result = set<Clause>();
            for (const k1 of cnfg) {
                for (const k2 of cnfh) {
                    result.add(k1.union(k2));
                }
            }
            return result;
        }
        case '∀': {
            const [_, x, g] = f;
            return cnf(g);
        }
        case '⚛️':
            return set<Clause>(set<Literal>(toTuple(f)));
        default:
            return set<Clause>(set<Literal>(toTuple(f)));
    }
}

In [48]:
console.dir(f7, { depth: null });

[
  '∀',
  'G',
  [
    '∀',
    'C',
    [
      '∀',
      'A',
      [
        '∧',
        [
          '∨',
          [ '¬', [ '⚛️', 'grandparent', 'G', 'C' ] ],
          [
            '∧',
            [ '⚛️', 'parent', 'G', [ 'sk1', 'G', 'C' ] ],
            [ '⚛️', 'parent', [ 'sk1', 'G', 'C' ], 'C' ]
          ]
        ],
        [
          '∨',
          [
            '∨',
            [ '¬', [ '⚛️', 'parent', 'G', 'A' ] ],
            [ '¬', [ '⚛️', 'parent', 'A', 'C' ] ]
          ],
          [ '⚛️', 'grandparent', 'G', 'C' ]
        ]
      ]
    ]
  ]
]


In [51]:
const f8 = cnf(f7);
console.log(f8.size);

3


## Putting Everything Together

In [52]:
function normalize(f: Formula): CNFSet {
    const f1 = eliminateBiconditional(f);
    const f2 = eliminateConditional(f1);
    const f3 = nnf(f2);
    const [Qs, f4] = extractQuantifiers(f3);
    const f5 = attachQuantifiers(Qs, f4);
    const f6 = skolemize(f5, []);
    return cnf(f6);
}

In [53]:
console.log(normalize(f1).size);

3


In [54]:
function prettify(M: CNFSet): string {
    if (M.size === 0) return '{}';
    let result = "{\n";
    const clauses = [...M];
    for (let i = 0; i < clauses.length; i++) {
        const A = clauses[i];
        if (A.size === 0) {
            result += "    {},\n";
        } else {
            result += "    {" + [...A].map(lit => lit.toString()).join(", ") + "}";
            if (i < clauses.length - 1) result += ",\n";
            else result += "\n";
        }
    }
    result += "}";
    return result;
}

In [55]:
function test(s: string): void {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    console.log(prettify(normalize(f)));
}

In [56]:
test(s);

The knf of ∀G:∀C:(grandparent(G, C) ↔ ∃P: (parent(G, P) ∧ parent(P, C))) is:
{
    {(¬, (Atom, grandparent, G, C)), (Atom, parent, G, (sk3, G, C))},
    {(¬, (Atom, grandparent, G, C)), (Atom, parent, (sk3, G, C), C)},
    {(¬, (Atom, parent, G, A)), (¬, (Atom, parent, A, C)), (Atom, grandparent, G, C)}
}


In [57]:
test('¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V))');

The knf of ¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V)) is:
{
    {(Atom, p, X, (sk4))},
    {(¬, (Atom, p, (sk5), V))}
}
